In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
np.set_printoptions(suppress=True)
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout, GRU, Bidirectional
from keras.optimizers import SGD
from keras.models import Sequential

from keras.callbacks import History
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard

In [2]:
technique_indicatorttm = np.load('D:/paper/onging/bankgraph/bank_stock_matrix_ttm.npy',allow_pickle=True)
technique_indicatorpbr = np.load('D:/paper/onging/bankgraph/bank_stock_matrix_pbr.npy',allow_pickle=True)
technique_indicatorps = np.load('D:/paper/onging/bankgraph/bank_stock_matrix_ps.npy',allow_pickle=True)
technique_indicatorpcf = np.load('D:/paper/onging/bankgraph/bank_stock_matrix_pcf.npy',allow_pickle=True)
sample_data = np.concatenate((technique_indicatorttm,technique_indicatorpbr,technique_indicatorps,technique_indicatorpcf),axis=2)
sample_data[pd.isna(sample_data)] = 5
sample_data.shape

(1215, 15, 12)

In [3]:
data = pd.read_csv('D:/paper/onging/data/bank/bankclose.csv',index_col=0)

In [4]:
data.shape
#data[pd.isna(data)] = 5
from sklearn.preprocessing import MinMaxScaler
data_scaler = MinMaxScaler(feature_range=(-1, 1))
data_dataset = data_scaler.fit_transform(data)
data_dataset.shape

(1215, 15)

In [5]:
p=1
Label_data_2 = data_dataset[p-1:,:]
Label_data_2.shape

(1215, 15)

In [6]:
#input 部分
input_data = sample_data[:,:,:]
input_data.shape

(1215, 15, 12)

In [7]:
input_data_final = np.zeros((input_data.shape[0]+1-p,p,input_data.shape[1],input_data.shape[2]))
input_data_final.shape

(1215, 1, 15, 12)

In [8]:
for i in range(input_data.shape[0]+1-p):
    one_line = []
    for j in range(p):
        one_line.append(input_data[i+j])
        
    input_data_final[i] = np.array(one_line)

In [ ]:
# 动态读取维度，避免硬编码导致后续消融实验 shape 不一致
N = input_data.shape[1]          # 股票数量
M = input_data_final.shape[0]    # 样本数量
P = p                            # look-back window
F = input_data.shape[2]          # 特征数量

TEST_SIZE = 0.3
RANDOM_STATE = 4

print("N, M, P, F =", N, M, P, F)


In [10]:
T = np.load('D:/paper/onging/bankgraph/shareholding_relation_adjacent_matrix_tongyi.npy')
I= np.load('D:/paper/onging/bankgraph/bank_qa_graph.npy')
S=np.load('D:/paper/onging/bankgraph/bankvolume_kendall.npy')
Fixed_Matrices = [S,I,T]

In [11]:
# T = np.random.normal(size=(N,N))
 #I = np.random.normal(size=(N,N))
 #S = np.random.normal(size=(N,N))
# I=T
 #S=T

In [12]:
samples = input_data_final#np.random.normal(size=(M, P, N, F))
labels = Label_data_2#np.random.normal(size=(M, N))

In [13]:
indices = np.arange(data.shape[0]).reshape(-1, 1)
label_with_index = np.hstack((indices, labels)) 
label_with_index

array([[   0.        ,   -0.37842779,    0.73466476, ...,    0.11597374,
           0.12359551,    0.06426735],
       [   1.        ,   -0.42352224,    0.71754636, ...,    0.09846827,
           0.14606742,    0.08997429],
       [   2.        ,   -0.43327239,    0.71754636, ...,    0.03719912,
           0.14606742,    0.10025707],
       ...,
       [1212.        ,   -0.44667885,   -0.81169757, ...,   -0.8380744 ,
          -0.79775281,   -0.62467866],
       [1213.        ,   -0.46008531,   -0.82881598, ...,   -0.84682713,
          -0.80898876,   -0.6503856 ],
       [1214.        ,   -0.44424132,   -0.81740371, ...,   -0.82932166,
          -0.79775281,   -0.62467866]])

In [ ]:
# ===============================
# Chronological split for forecasting
# ===============================

dates = pd.to_datetime(data.index[p-1:p-1+len(samples)])

labels = Label_data_2
samples = input_data_final

train_mask = (dates >= '2018-01-02') & (dates <= '2021-12-31')
test_mask  = (dates >= '2022-01-01') & (dates <= '2022-12-31')

# 从训练集最后20%切出validation，避免随机打乱
train_indices = np.where(train_mask)[0]
test_indices = np.where(test_mask)[0]

val_size = int(len(train_indices) * 0.2)
val_indices = train_indices[-val_size:]
real_train_indices = train_indices[:-val_size]

x_train = samples[real_train_indices]
y_train1 = labels[real_train_indices]

x_val = samples[val_indices]
y_val1 = labels[val_indices]

x_test = samples[test_indices]
y_test1 = labels[test_indices]

x_train = tf.constant(x_train, dtype=tf.float32)
y_train1 = tf.constant(y_train1, dtype=tf.float32)
x_val = tf.constant(x_val, dtype=tf.float32)
y_val1 = tf.constant(y_val1, dtype=tf.float32)
x_test = tf.constant(x_test, dtype=tf.float32)
y_test1 = tf.constant(y_test1, dtype=tf.float32)

print(x_train.shape, y_train1.shape)
print(x_val.shape, y_val1.shape)
print(x_test.shape, y_test1.shape)

In [ ]:
# 注意：时间顺序划分后，变量名已经是 y_train1/y_val1/y_test1。
# 原来的 y_train/y_val/y_test 不再存在，因此不要再运行旧代码。
# 如需反归一化，可在预测后使用：
# y_pred_raw = data_scaler.inverse_transform(y_pred)
# y_test_raw = data_scaler.inverse_transform(y_test1.numpy())


In [17]:
import tensorflow as tf
from tensorflow.keras.layers import MultiHeadAttention, Dense, LayerNormalization, Dropout

In [18]:
class GNNTransformer(tf.keras.Model):
    def __init__(self, N, F, Units_GNN, Units_Transformer, Units_FC,
                 Fixed_Matrices, Matrix_Weights, Is_Dyn,
                 num_heads,
                 dff,
                 dropout_rate=0.1,
                 kernel_initializer='glorot_uniform',
                 bias_initializer='zeros'):
        super(GNNTransformer, self).__init__()
        # Number of Nodes
        self.N = N
        # the number of input features
        self.F = F
        # pre-defined matrices: shape=[(N,N),(N,N),(N,N)]
        self.mat = Fixed_Matrices
        
        if Is_Dyn:
            self.mats = self.dyn
        else:
            self.mats = self.add_weight(name='w_Matrices', shape=(self.N, self.N),
                                        initializer=tf.keras.initializers.get(kernel_initializer),
                                        trainable=True)
            for i in range(len(Matrix_Weights)):
                coe = tf.Variable(1.0, trainable=True)
                self.mats = self.mats + Matrix_Weights[i] * self.mat[i] * coe
        
        # GNN Weights
        self.units_gnn = Units_GNN
        self.w_gnn = []
        self.b_gnn = []
        pre = self.F
        for i in range(len(self.units_gnn)):
            aft = self.units_gnn[i]
            w = self.add_weight(name='w_GNN', shape=(pre, aft),
                                initializer=tf.keras.initializers.get(kernel_initializer),
                                trainable=True)
            self.w_gnn.append(w)
            b = self.add_weight('b_GNN', shape=(aft,),
                                initializer=tf.keras.initializers.get(bias_initializer),
                                trainable=True)
            self.b_gnn.append(b)
            pre = aft
        
        # Transformer parameters
        self.units_transformer = Units_Transformer
        self.num_heads = num_heads
        self.dff = dff
        self.dropout_rate = dropout_rate

        # Transformer layers
        self.attention_layers = []
        self.dense_proj_layers = []
        self.norm_layers = []
        self.dropout_layers = []
        for _ in range(len(self.units_transformer)):
            self.attention_layers.append(MultiHeadAttention(num_heads=self.num_heads, key_dim=self.units_transformer[0]))
            self.dense_proj_layers.append(tf.keras.Sequential([
                Dense(self.dff, activation='relu'),
                Dense(self.units_transformer[0])
            ]))
            self.norm_layers.append(LayerNormalization(epsilon=1e-6))
            self.dropout_layers.append(Dropout(self.dropout_rate))
        
        # FC_weights
        self.units_fc = Units_FC
        self.w_fc = []
        self.b_fc = []
        pre = self.units_transformer[-1]
        for i in range(len(self.units_fc)):
            aft = self.units_fc[i]
            w = self.add_weight(name='w_FC', shape=(pre, aft),
                                initializer=tf.keras.initializers.get(kernel_initializer),
                                trainable=True)
            self.w_fc.append(w)
            b = self.add_weight(name='b_FC', shape=(aft,),
                                initializer=tf.keras.initializers.get(bias_initializer),
                                trainable=True)
            self.b_fc.append(b)
            pre = aft
    
    def Multi_GNN(self, inputs):
        '''
        inputs: shape=(None, P, N, F)
        x_gnn: shape=(None, P, N, C)
        '''
        P = inputs.shape[1]
        x_gnn = []
        for t in range(P):
            # (None, P, N, F) => (None, N, F)
            xt_gnn = inputs[:, t, :, :]
            
            # (N, N) * (None, N, F) * (F, C) => (None, N, C)
            for i in range(len(self.units_gnn)):
                xt_gnn = self.mats @ xt_gnn @ self.w_gnn[i] + self.b_gnn[i]
                xt_gnn = tf.nn.tanh(xt_gnn)
            x_gnn.append(xt_gnn)
        # (None, P, N, C)
        x_gnn = tf.stack(x_gnn, axis=1)
        return x_gnn
    
    def Transformer(self, x, x_gcn):
        '''
        x:shape=(None,P,N,F)
        x_gcn:shape=(None,P,N,C)
        x_transformer:shape=(None,N,G)
        '''
        # (None, P, N, F+C)
        x = tf.concat([x, x_gcn], axis=-1)

        for attention_layer, dense_proj_layer, norm_layer, dropout_layer in zip(self.attention_layers, self.dense_proj_layers, self.norm_layers, self.dropout_layers):
            attn_output = attention_layer(x, x, x)
            attn_output = dropout_layer(attn_output)
            x = norm_layer(x + attn_output)

            proj_output = dense_proj_layer(x)
            proj_output = dropout_layer(proj_output)
            x = norm_layer(x + proj_output)

        # (None, N, G)
        x_transformer = x[:, -1, :, :]
        return x_transformer
    
    def FC(self, x_transformed):
        '''
        x_transformed:shape=(None, N, units_transformer[-1])
        outputs: shape=(None, N, 1)
        '''
        x = x_transformed
        for i in range(len(self.w_fc)):
            x = x @ self.w_fc[i] + self.b_fc[i]
            x = tf.nn.sigmoid(x)
        # (None, N)
        x_fc = tf.squeeze(x, axis=-1)
        return x_fc

    def build(self, input_shape):
        super().build(input_shape)
        
    def call(self, inputs):
        '''
        inputs: shape=(None, P, N, F)
        x_fc: shape=(None, N)
        '''
        # (None, P, N, C)
        x_gnn = self.Multi_GNN(inputs)
        # (None, N, G)
        x_transformer = self.Transformer(inputs, x_gnn)
        # (None, N)
        x_fc = self.FC(x_transformer)
        return x_fc



In [ ]:
class AblationGNNTransformer(tf.keras.Model):
    """
    Ablation model for MGT:
    - use_gnn=False: Transformer-only baseline
    - use_transformer=False: MGNN-only baseline
    - learnable_alpha=False: fixed relation fusion baseline

    Input:  [B, P, N, F]
    Output: [B, N]
    """
    def __init__(self, N, F, Units_GNN, Fixed_Matrices, Matrix_Weights,
                 use_gnn=True,
                 use_transformer=True,
                 learnable_alpha=True,
                 num_heads=4,
                 dff=128,
                 dropout_rate=0.1):
        super().__init__()

        self.N = N
        self.F = F
        self.use_gnn = use_gnn
        self.use_transformer = use_transformer

        self.Fixed_Matrices = [tf.constant(A, dtype=tf.float32) for A in Fixed_Matrices]
        self.Matrix_Weights = tf.constant(Matrix_Weights, dtype=tf.float32)

        self.Units_GNN = Units_GNN
        self.C = Units_GNN[-1] if use_gnn else 0
        self.model_dim = F + self.C if use_gnn else F

        # 只有在使用 GNN 时才创建邻接矩阵参数，避免 Transformer-only 出现无梯度变量
        if self.use_gnn:
            self.WM = self.add_weight(
                name="base_adjacency",
                shape=(N, N),
                initializer="glorot_uniform",
                trainable=True
            )
            self.alpha = self.add_weight(
                name="relation_alpha",
                shape=(len(Fixed_Matrices),),
                initializer="ones",
                trainable=learnable_alpha
            )

            self.gnn_layers = []
            for u in Units_GNN:
                self.gnn_layers.append(tf.keras.layers.Dense(u, activation="tanh"))

        if self.use_transformer:
            key_dim = max(1, self.model_dim // num_heads)
            self.attn = tf.keras.layers.MultiHeadAttention(
                num_heads=num_heads,
                key_dim=key_dim
            )
            self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
            self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
            self.ffn = tf.keras.Sequential([
                tf.keras.layers.Dense(dff, activation="relu"),
                tf.keras.layers.Dense(self.model_dim)
            ])
            self.dropout = tf.keras.layers.Dropout(dropout_rate)

        self.out = tf.keras.layers.Dense(1, activation="linear")

    def effective_adjacency(self):
        A = self.WM
        for i, M in enumerate(self.Fixed_Matrices):
            A = A + self.Matrix_Weights[i] * self.alpha[i] * M
        return A

    def multi_gnn(self, x):
        # x: [B, P, N, F]
        A = self.effective_adjacency()
        outs = []
        for x_t in tf.unstack(x, axis=1):  # each x_t: [B, N, F]
            h = x_t
            for layer in self.gnn_layers:
                h = tf.einsum("ij,bjf->bif", A, h)
                h = layer(h)
            outs.append(h)
        return tf.stack(outs, axis=1)      # [B, P, N, C]

    def temporal_transformer_per_asset(self, x, training=False):
        # x: [B, P, N, D]
        # 对每只股票单独沿时间轴做 self-attention: [B*N, P, D]
        B = tf.shape(x)[0]
        P_ = tf.shape(x)[1]
        N_ = tf.shape(x)[2]
        D_ = tf.shape(x)[3]

        x_bn = tf.transpose(x, perm=[0, 2, 1, 3])       # [B, N, P, D]
        x_bn = tf.reshape(x_bn, [B * N_, P_, D_])       # [B*N, P, D]

        attn_out = self.attn(x_bn, x_bn, x_bn, training=training)
        x_bn = self.norm1(x_bn + self.dropout(attn_out, training=training))

        ffn_out = self.ffn(x_bn, training=training)
        x_bn = self.norm2(x_bn + self.dropout(ffn_out, training=training))

        x_bn = tf.reshape(x_bn, [B, N_, P_, D_])        # [B, N, P, D]
        x = tf.transpose(x_bn, perm=[0, 2, 1, 3])       # [B, P, N, D]
        return x

    def call(self, inputs, training=False):
        x = inputs

        if self.use_gnn:
            x_gnn = self.multi_gnn(inputs)
            x = tf.concat([inputs, x_gnn], axis=-1)

        if self.use_transformer:
            x = self.temporal_transformer_per_asset(x, training=training)

        # last-step pooling: [B, N, D]
        x_last = x[:, -1, :, :]

        # [B, N]
        y_hat = self.out(x_last)
        y_hat = tf.squeeze(y_hat, axis=-1)
        return y_hat


In [ ]:
# 确认三张图的顺序：Fixed_Matrices = [KD, QA, SH]
# 前面 Cell 9 已经定义：S=KD, I=QA, T=SH
Fixed_Matrices = [S, I, T]

print("Matrix order: [KD, QA, SH]")
for idx, A in enumerate(Fixed_Matrices):
    print(idx, np.asarray(A).shape, np.nanmin(A), np.nanmax(A))


In [ ]:
# 不要再写 S = KD, I = QA, T = SH，当前 notebook 中 KD/QA/SH 变量未定义。
# 使用 Cell 9 已经加载好的 S, I, T 即可。


In [ ]:
ABLATION_CONFIGS = {
    "A0_Full_MGT": {
        "Matrix_Weights": [1, 1, 1],
        "use_gnn": True,
        "use_transformer": True,
        "learnable_alpha": True
    },
    "A1_Transformer_only": {
        "Matrix_Weights": [0, 0, 0],
        "use_gnn": False,
        "use_transformer": True,
        "learnable_alpha": False
    },
    "A2_MGNN_only": {
        "Matrix_Weights": [1, 1, 1],
        "use_gnn": True,
        "use_transformer": False,
        "learnable_alpha": True
    },
    "A3_SH_only": {
        "Matrix_Weights": [0, 0, 1],
        "use_gnn": True,
        "use_transformer": True,
        "learnable_alpha": True
    },
    "A4_QA_only": {
        "Matrix_Weights": [0, 1, 0],
        "use_gnn": True,
        "use_transformer": True,
        "learnable_alpha": True
    },
    "A5_KD_only": {
        "Matrix_Weights": [1, 0, 0],
        "use_gnn": True,
        "use_transformer": True,
        "learnable_alpha": True
    },
    "A6_SH_QA": {
        "Matrix_Weights": [0, 1, 1],
        "use_gnn": True,
        "use_transformer": True,
        "learnable_alpha": True
    },
    "A7_SH_KD": {
        "Matrix_Weights": [1, 0, 1],
        "use_gnn": True,
        "use_transformer": True,
        "learnable_alpha": True
    },
    "A8_QA_KD": {
        "Matrix_Weights": [1, 1, 0],
        "use_gnn": True,
        "use_transformer": True,
        "learnable_alpha": True
    },
    "A9_Fixed_fusion": {
        "Matrix_Weights": [1, 1, 1],
        "use_gnn": True,
        "use_transformer": True,
        "learnable_alpha": False
    }
}

In [ ]:
import os
import random
from sklearn.metrics import mean_squared_error, mean_absolute_error

ABLATION_DIR = "D:/paper/onging/ablation"
os.makedirs(ABLATION_DIR, exist_ok=True)

def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)

def _to_numpy(x):
    return x.numpy() if hasattr(x, "numpy") else np.asarray(x)

def train_one_ablation(name, cfg, seed=1):
    set_seed(seed)

    model = AblationGNNTransformer(
        N=N,
        F=F,
        Units_GNN=[16, 32, 64],
        Fixed_Matrices=Fixed_Matrices,
        Matrix_Weights=cfg["Matrix_Weights"],
        use_gnn=cfg["use_gnn"],
        use_transformer=cfg["use_transformer"],
        learnable_alpha=cfg["learnable_alpha"],
        num_heads=4,
        dff=128,
        dropout_rate=0.1
    )

    # 先用 dummy input 触发变量创建，避免 subclass model 的 build/count_params 问题
    _ = model(tf.zeros((1, P, N, F), dtype=tf.float32), training=False)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=[
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
            tf.keras.metrics.MeanAbsoluteError(name="mae")
        ]
    )

    ckpt_path = os.path.join(ABLATION_DIR, f"{name}_seed{seed}.weights.h5")

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=ckpt_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=0
        )
    ]

    history = model.fit(
        x_train, y_train1,
        validation_data=(x_val, y_val1),
        epochs=100,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )

    y_pred = model.predict(x_test)
    y_true = _to_numpy(y_test1)

    mse = mean_squared_error(y_true.reshape(-1), y_pred.reshape(-1))
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true.reshape(-1), y_pred.reshape(-1))

    result = {
        "model": name,
        "seed": seed,
        "params": model.count_params(),
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "best_val_loss": np.min(history.history["val_loss"])
    }

    # 保存预测结果和真实值，方便后续画图与接入 FinRL
    pred_path = os.path.join(ABLATION_DIR, f"{name}_seed{seed}_pred.csv")
    true_path = os.path.join(ABLATION_DIR, f"{name}_seed{seed}_true.csv")
    hist_path = os.path.join(ABLATION_DIR, f"{name}_seed{seed}_history.csv")

    pd.DataFrame(y_pred).to_csv(pred_path, index=False)
    pd.DataFrame(y_true).to_csv(true_path, index=False)
    pd.DataFrame(history.history).to_csv(hist_path, index=False)

    return result


In [ ]:
all_results = []

SEEDS = [1, 2, 3]

for name, cfg in ABLATION_CONFIGS.items():
    for seed in SEEDS:
        print(f"\n========== Running {name}, seed={seed} ==========")
        result = train_one_ablation(name, cfg, seed)
        all_results.append(result)

results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(ABLATION_DIR, "forecast_ablation_raw_results.csv"), index=False)

summary_df = results_df.groupby("model").agg({
    "params": "mean",
    "mse": ["mean", "std"],
    "rmse": ["mean", "std"],
    "mae": ["mean", "std"],
    "best_val_loss": ["mean", "std"]
})

summary_df.to_csv(os.path.join(ABLATION_DIR, "forecast_ablation_summary.csv"))
summary_df

旧版单模型训练/分类代码已停用，避免与消融实验冲突。

请不要再运行原来的 categorical_crossentropy、SGD、val_loss8、y_test/y_train 相关代码。
当前消融实验的主入口是前面的 `train_one_ablation()` 和 `ABLATION_CONFIGS`。


旧版单模型训练/分类代码已停用，避免与消融实验冲突。

请不要再运行原来的 categorical_crossentropy、SGD、val_loss8、y_test/y_train 相关代码。
当前消融实验的主入口是前面的 `train_one_ablation()` 和 `ABLATION_CONFIGS`。


旧版单模型训练/分类代码已停用，避免与消融实验冲突。

请不要再运行原来的 categorical_crossentropy、SGD、val_loss8、y_test/y_train 相关代码。
当前消融实验的主入口是前面的 `train_one_ablation()` 和 `ABLATION_CONFIGS`。


旧版单模型训练/分类代码已停用，避免与消融实验冲突。

请不要再运行原来的 categorical_crossentropy、SGD、val_loss8、y_test/y_train 相关代码。
当前消融实验的主入口是前面的 `train_one_ablation()` 和 `ABLATION_CONFIGS`。


旧版单模型训练/分类代码已停用，避免与消融实验冲突。

请不要再运行原来的 categorical_crossentropy、SGD、val_loss8、y_test/y_train 相关代码。
当前消融实验的主入口是前面的 `train_one_ablation()` 和 `ABLATION_CONFIGS`。


旧版单模型训练/分类代码已停用，避免与消融实验冲突。

请不要再运行原来的 categorical_crossentropy、SGD、val_loss8、y_test/y_train 相关代码。
当前消融实验的主入口是前面的 `train_one_ablation()` 和 `ABLATION_CONFIGS`。


旧版单模型训练/分类代码已停用，避免与消融实验冲突。

请不要再运行原来的 categorical_crossentropy、SGD、val_loss8、y_test/y_train 相关代码。
当前消融实验的主入口是前面的 `train_one_ablation()` 和 `ABLATION_CONFIGS`。
